In [114]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [115]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["fe01__family"]

# Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# Uncomment to run all experiments and update results.
# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     result_df = save_results(exp_result)
#     save_configs(exp_config)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#     results_df=result_df,
#     reference_group="baseline__raw",
#     compare_groups=[Name],
#     metrics=["test_accuracy_mean"],
# )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe09__ticket_group_size


In [116]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [117]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
)

print("Workflow completed. Here are the results:")
print("Comparison between baseline and feature engineering group:")
print(workflow["comparison"])
print("Summary of comparison:")
print(workflow["summary"])
print("Leaderboard:")
print(workflow["leaderboard"])

running exp: {'name': 'fe01__family__logreg', 'features': ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone'], 'feature_engineering': [<function add_family_features at 0x000001BCC37048B0>], 'preprocessing': {'numeric_features': ['Age', 'Fare', 'FamilySize', 'IsAlone'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 01: replaces SibSp/Parch with FamilySize and IsAlone.', 'stage': 'fe01', 'feature_group': 'family', 'group': 'fe01__family'}
running exp: {'name': 'fe01__family__knn', 'features': ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone'], 'feature_engineering': [<function ad

In [118]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()


Full workflow report:

Report
### fe01__family

_Description pending._

<details>
<summary>Conclusion</summary>

<details>
<summary>Experiment details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group   | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe01__family    | logreg        |                          0.786 |                        0.795 |                      0.009 |                    0.713 |                  0.721 |                0.008 |
| baseline__raw     | fe01__family    | knn           |                          0.809 |                        0.805 |             

In [119]:
import pprint
Feature_effect = analyze_feature_effect(workflow['comparison'])
pprint.pprint(Feature_effect)

[{'compare_group': 'fe01__family',
  'discard': False,
  'max_delta': np.float64(0.009),
  'mean_delta': np.float64(0.0),
  'metric': 'test_accuracy_mean',
  'min_delta': np.float64(-0.006),
  'negative_models': ['knn', 'random_forest'],
  'neutral_models': ['svc', 'extra_trees', 'xgb'],
  'positive_models': ['logreg', 'decision_tree'],
  'recommended_for_all': False,
  'recommended_models': ['logreg', 'decision_tree'],
  'verdict': 'model_specific_mixed'}]


In [120]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [121]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [122]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [123]:
# print(workflow["all_results"])

In [124]:
for model in MODEL_REGISTRY:
    model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
    print(f"Model progression for {model}:")
    print(model_progression_df)
    print()

Model progression for logreg:
      stage                feature_group  test_accuracy_mean
0  baseline                          raw               0.786
1      fe01                       family               0.795
2      fe02                    has_cabin               0.791
3      fe03                         deck               0.791
4      fe04               cabin_features               0.791
5      fe05                        title               0.825
6      fe06         age_imputation_title               0.787
7      fe07  age_imputation_title_pclass               0.799
8      fe08       fare_per_family_member               0.789
9      fe09            ticket_group_size               0.788

Model progression for knn:
      stage                feature_group  test_accuracy_mean
0  baseline                          raw               0.809
1      fe01                       family               0.805
2      fe02                    has_cabin               0.809
3      fe03                

In [125]:
bins = [0, 14, 35, 60, 100]
labels = ['0', '2', '3', '1']
exp_df['Age_bin'] = pd.cut(exp_df['Age'], bins=bins, labels=labels, right=False)

NameError: name 'exp_df' is not defined

In [ ]:
for age in range(0,18):
    test_df = exp_df[exp_df['Age'] == age][['Age','Survived']]
    test_df = test_df.groupby('Survived').value_counts()
    print(test_df)

Series([], Name: count, dtype: int64)
Survived  Age
0         1.0    2
1         1.0    5
Name: count, dtype: int64
Survived  Age
0         2.0    7
1         2.0    3
Name: count, dtype: int64
Survived  Age
0         3.0    1
1         3.0    5
Name: count, dtype: int64
Survived  Age
0         4.0    3
1         4.0    7
Name: count, dtype: int64
Survived  Age
1         5.0    4
Name: count, dtype: int64
Survived  Age
0         6.0    1
1         6.0    2
Name: count, dtype: int64
Survived  Age
0         7.0    2
1         7.0    1
Name: count, dtype: int64
Survived  Age
0         8.0    2
1         8.0    2
Name: count, dtype: int64
Survived  Age
0         9.0    6
1         9.0    2
Name: count, dtype: int64
Survived  Age 
0         10.0    2
Name: count, dtype: int64
Survived  Age 
0         11.0    3
1         11.0    1
Name: count, dtype: int64
Survived  Age 
1         12.0    1
Name: count, dtype: int64
Survived  Age 
1         13.0    2
Name: count, dtype: int64
Survived  Age 
